# LLM-as-a-Judge Filter

Notebook to apply a basic LLM-as-a-Judge filter on 3 datasets:
- ASQA test set (newly created by splitting the original ASQA train set)
- MS Macro (500 longest answers from the original test set)
- WikiEval 

This is to compare the performance of the filter with RAGAS filter.


## Imports and setup    

In [25]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
import json
from dotenv import load_dotenv

import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score


from IPython.display import display

sys.path.append('..')

from src.filtering.llm_judge_filter import LLMJudgeFilter



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# ============================================================
# Paths
# ============================================================

DATA_PATH = Path('../data/labeled_merged_test.csv')

OUTPUT_DIR = Path('../results/llm_filter')
OUTPUT_FILTER = OUTPUT_DIR / 'classification'
OUTPUT_FILTER.mkdir(parents=True, exist_ok=True)

print(f'Output dir: {OUTPUT_FILTER}')

Output dir: ../results/llm_filter/classification


## Imports data

In [27]:
# ============================================================
# Quick schema check
# ============================================================

REQUIRED_COLS = {'id', 'question', 'answer', 'context'}
OPTIONAL_EVAL_COLS = {'label', 'gold_ans', 'gold_answer', 'reference', 'reference_answer', 'ground_truth'}

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

display(df.head())

# count sample per dataset
dataset_counts = df['dataset'].value_counts()
print("Sample counts per dataset:")
display(dataset_counts)

Dataset: ../data/labeled_merged_test.csv
Shape: (1962, 6)
Columns: ['id', 'question', 'context', 'answer', 'label', 'dataset']


,id,question,context,answer,label,dataset
0,asqa_2288_hallu,When did the first indian expedition to antarc...,- (Indian Antarctic Program) The first Indian ...,The first Indian expedition led by an Indian t...,0,asqa
1,14_pos,"When did the Inter Expo Center in Sofia, Bulga...",The Inter Expo Center (IEC) is a multi-purpose...,"The Inter Expo Center in Sofia, Bulgaria opene...",1,wikieval
2,asqa_4328,When was the first mobile phone text message s...,- (QA_1) When was the first mobile phone text ...,SMS messaging was used for the first time on 3...,1,asqa
3,asqa_3065,Who sings you'll be a woman soon?,"- (Girl, You'll Be a Woman Soon) Girl, You'll ...","Girl, You'll Be a Woman Soon is a song written...",1,asqa
4,asqa_3132,When does hotel transylvania part 3 come out?,- (Hotel Transylvania 3: Summer Vacation) Hote...,"Hotel Transylvania 3, released internationally...",1,asqa


Sample counts per dataset:


dataset
asqa        1737
msmarco      207
wikieval      18
Name: count, dtype: int64

## Initialize LLM Judge Filter & Evaluator

In [28]:
# ============================================================
# Init LLM Judge
# ============================================================

# LLMJudgeFilter sẽ đọc OPENAI_API_KEY từ environment.

judge = LLMJudgeFilter(
    model='gpt-4o-mini',
    api_key=OPENAI_API_KEY,
    output_dir=OUTPUT_FILTER,
    temperature=0.0,
    max_retries=3,
    sleep_between_retries=2.0,
    max_context_chars=12000,
)

In [29]:
# ============================================================
# Run LLM judge on one dataset
# ============================================================

def run_llm_judge_dataset(
    run_time: int,
    df: pd.DataFrame,
    eval_mode: str = 'classification',
    resume: bool = True,
    save_every: int = 1,
):
    print(f'Running LLM judge {run_time}')
    print(f'Input: {df.shape[0]} samples')

    dataset_name_mapping = {
        "asqa": "ASQA",
        "msmarco": "MS MARCO",
        "wikieval": "WikiEval",
    }

    metrics_dfs = []
    for i in range(run_time):
        print('=' * 80)
        print(f'Run {i+1}/{run_time}')
        out = judge.run(
            data=df,
            output_path=OUTPUT_FILTER / f'predictions_run_{i+1}.csv',
            eval_mode=eval_mode,
            resume=resume,
            save_every=save_every,
        )
        pred_df = pd.read_csv(OUTPUT_FILTER / f'predictions_run_{i+1}.csv')
        # Split to datasets
        metric_df = []
        for dataset_name, group in pred_df.groupby('dataset'):
            acc = accuracy_score(group['label'], group['filter_label'])
            num_samples = group.shape[0]
            accepted = (group['filter_label'] == 1).sum()
            acceptance_rate = accepted / num_samples if num_samples > 0 else 0.0
            prec = precision_score(group['label'], group['filter_label'], zero_division=0)
            red = recall_score(group['label'], group['filter_label'], zero_division=0)
            f1 = f1_score(group['label'], group['filter_label'], zero_division=0)
            roc_auc = roc_auc_score(group['label'], group['filter_label']) if len(group['label'].unique()) > 1 else float('nan')
            conf = group['filter_confidence'].mean() if 'filter_confidence' in group.columns else float('nan')      
            metric_df.append({
                "dataset": dataset_name,
                "num_samples": num_samples,
                "accuracy": acc,
                "acceptance_rate": acceptance_rate,
                "precision": prec,
                "recall": red,
                "f1": f1,
                "roc_auc": roc_auc,
                "avg_confidence": conf,
            }
            )
        # add overall dataset metrics
        overall_acc = accuracy_score(pred_df['label'], pred_df['filter_label'])
        overall_num_samples = pred_df.shape[0]
        overall_accepted = (pred_df['filter_label'] == 1).sum()
        overall_acceptance_rate = overall_accepted / overall_num_samples if overall_num_samples > 0 else 0.0
        overall_prec = precision_score(pred_df['label'], pred_df['filter_label'], zero_division=0)
        overall_red = recall_score(pred_df['label'], pred_df['filter_label'], zero_division=0)
        overall_f1 = f1_score(pred_df['label'], pred_df['filter_label'], zero_division=0)
        overall_roc_auc = roc_auc_score(pred_df['label'], pred_df['filter_label']) if len(pred_df['label'].unique()) > 1 else float('nan')
        overall_conf = pred_df['filter_confidence'].mean() if 'filter_confidence' in pred_df.columns else float('nan')
        metric_df.append({
            "dataset": "Overall",
            "num_samples": overall_num_samples,
            "accuracy": overall_acc,
            "acceptance_rate": overall_acceptance_rate,
            "precision": overall_prec,
            "recall": overall_red,
            "f1": overall_f1,
            "roc_auc": overall_roc_auc,
            "avg_confidence": overall_conf,
        })
        metric_df = pd.DataFrame(metric_df)
        metric_df["dataset"] = metric_df["dataset"].map(dataset_name_mapping).fillna(metric_df["dataset"])
        metrics_dfs.append(metric_df)
        display(metric_df.sort_values("dataset", ascending=True).reset_index(drop=True))

    
    # avg per dataset
    avg_metrics_df = pd.concat(metrics_dfs).groupby("dataset").mean().reset_index()


    all_metrics_df = pd.concat(metrics_dfs, ignore_index=True)

    stat_df = (
        all_metrics_df
        .groupby("dataset")
        .agg({
            "accuracy": ["mean", "std"],
            "precision": ["mean", "std"],
            "recall": ["mean", "std"],
            "f1": ["mean", "std"],
            "roc_auc": ["mean", "std"],
            "acceptance_rate": ["mean", "std"],
            "avg_confidence": ["mean", "std"],
        })
    )
                                                                

    return avg_metrics_df, stat_df

## Run evaluation on Merged Test Set
Include samples fronm 3 datasets:
- ASQA
- MS MARCO
- WikiEval

In [30]:
summary_df, stat_df = run_llm_judge_dataset(
    run_time=3,
    df=df,
    eval_mode='classification',
    resume=True,
    save_every=1,
)

Running LLM judge 3
Input: 1962 samples
Run 1/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_2332_hallu', 'asqa_2639_hallu', 'asqa_2190_hallu', '11505', 'asqa_2741_hallu']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1,roc_auc,avg_confidence
0,ASQA,1737,0.846862,0.400115,0.935252,0.746269,0.830140,0.847153,0.915630
1,MS MARCO,207,0.917874,0.429952,0.988764,0.846154,0.911917,0.918223,0.897826
2,Overall,1962,0.855250,0.405199,0.940881,0.759391,0.840449,0.855642,0.913761
3,WikiEval,18,0.944444,0.611111,0.909091,1.000000,0.952381,0.937500,0.916667


Run 2/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_2332_hallu', 'asqa_2639_hallu', 'asqa_2190_hallu', '11505', 'asqa_2741_hallu']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1,roc_auc,avg_confidence
0,ASQA,1737,0.844560,0.398964,0.933622,0.742824,0.827366,0.844853,0.916436
1,MS MARCO,207,0.922705,0.434783,0.988889,0.855769,0.917526,0.923030,0.898792
2,Overall,1962,0.853211,0.405199,0.938365,0.757360,0.838202,0.853603,0.914653
3,WikiEval,18,0.888889,0.666667,0.833333,1.000000,0.909091,0.875000,0.925000


Run 3/3
Found 1962 completed samples. Resuming from last checkpoint.
last 5 completed ids: ['asqa_2332_hallu', 'asqa_2639_hallu', 'asqa_2190_hallu', '11505', 'asqa_2741_hallu']


LLM Judge (gpt-4o-mini): 0sample [00:00, ?sample/s]


,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1,roc_auc,avg_confidence
0,ASQA,1737,0.846287,0.397237,0.937681,0.742824,0.828956,0.846585,0.915774
1,MS MARCO,207,0.913043,0.425121,0.988636,0.836538,0.906250,0.913415,0.899758
2,Overall,1962,0.853721,0.402650,0.941772,0.755330,0.838310,0.854124,0.914169
3,WikiEval,18,0.888889,0.666667,0.833333,1.000000,0.909091,0.875000,0.925000


In [31]:
display(summary_df.sort_values("dataset", ascending=True).reset_index(drop=True))

summary_df.to_csv(OUTPUT_FILTER / 'llm_judge_summary.csv', index=False)
print(f"Saved average dataset metrics to: {OUTPUT_FILTER / 'llm_judge_summary.csv'}")

print("Statistics across runs:")
display(stat_df)

,dataset,num_samples,accuracy,acceptance_rate,precision,recall,f1,roc_auc,avg_confidence
0,ASQA,1737.0,0.845903,0.398772,0.935518,0.743972,0.828821,0.846197,0.915947
1,MS MARCO,207.0,0.917874,0.429952,0.988763,0.846154,0.911898,0.918223,0.898792
2,Overall,1962.0,0.854060,0.404349,0.940339,0.757360,0.838987,0.854456,0.914195
3,WikiEval,18.0,0.907407,0.648148,0.858586,1.000000,0.923521,0.895833,0.922222


Saved average dataset metrics to: ../results/llm_filter/classification/llm_judge_summary.csv
Statistics across runs:


accuracy           precision              recall            \
              mean       std      mean       std      mean       std   
dataset                                                                
ASQA      0.845903  0.001198  0.935518  0.002043  0.743972  0.001989   
MS MARCO  0.917874  0.004831  0.988763  0.000126  0.846154  0.009615   
Overall   0.854060  0.001061  0.940339  0.001767  0.757360  0.002030   
WikiEval  0.907407  0.032075  0.858586  0.043739  1.000000  0.000000   

                f1             roc_auc           acceptance_rate            \
              mean       std      mean       std            mean       std   
dataset                                                                      
ASQA      0.828821  0.001392  0.846197  0.001198        0.398772  0.001449   
MS MARCO  0.911898  0.005638  0.918223  0.004808        0.429952  0.004831   
Overall   0.838987  0.001267  0.854456  0.001059        0.404349  0.001471   
WikiEval  0.923521  0.024994  0.895833  0.036084        0.648148  0.032075   

         avg_confidence            
                   mean       std  
dataset                            
ASQA           0.915947  0.000430  
MS MARCO       0.898792  0.000966  
Overall        0.914195  0.000447  
WikiEval       0.922222  0.004811